# POCD-KansformerEPI — Vertex AI Training Guide

**Machine**: g2-standard-16 (1× NVIDIA L4, 16 vCPUs, 64 GB RAM)  
**Training**: 50 epochs with early stopping  
**Data**: BENGI benchmarks + pre-processed epigenetic .pt signals + optional hg19 reference genome

---

## Step 0 — Upload Data

Before running this notebook, upload these zip files via JupyterLab UI (`Upload` button):

1. **`POCD-KansformerEPI.zip`** — project code  
2. **`processed.zip`** — 48 `.pt` epigenetic signal files  
3. **`kansformer.zip`** — contains BENGI TSV benchmarks + config JSON  

Upload them to the home directory (`/home/jupyter/`).

## Step 1 — Unzip & Arrange Files

In [ ]:
import os
os.chdir(os.path.expanduser('~'))
print('Working dir:', os.getcwd())

In [ ]:
# Unzip project code
!unzip -qo POCD-KansformerEPI.zip -d POCD-KansformerEPI_tmp

# Handle nested folder (zip may contain POCD-KansformerEPI/POCD-KansformerEPI/)
import shutil, glob
src_candidates = glob.glob('POCD-KansformerEPI_tmp/POCD-KansformerEPI*')
if src_candidates:
    src = src_candidates[0]
else:
    src = 'POCD-KansformerEPI_tmp'

PROJECT = os.path.expanduser('~/POCD-KansformerEPI')
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
shutil.copytree(src, PROJECT)
shutil.rmtree('POCD-KansformerEPI_tmp', ignore_errors=True)
print('Project at:', PROJECT)
os.chdir(PROJECT)
!ls -la

In [ ]:
# Unzip kansformer data (has BENGI TSVs + genomic_data config)
os.chdir(os.path.expanduser('~'))
!unzip -qo kansformer.zip -d kansformer_tmp

# Find the src/data directory inside the kansformer zip
import glob
bengi_candidates = glob.glob('kansformer_tmp/**/BENGI', recursive=True)
data_root = os.path.dirname(bengi_candidates[0]) if bengi_candidates else 'kansformer_tmp'
print(f'Found data root: {data_root}')
!ls {data_root}

In [ ]:
# Set up data directory structure that train.py expects
import shutil

DATA_DIR = os.path.join(PROJECT, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

# Copy BENGI benchmarks
bengi_src = os.path.join(data_root, 'BENGI')
bengi_dst = os.path.join(DATA_DIR, 'BENGI')
if os.path.exists(bengi_src):
    if os.path.exists(bengi_dst):
        shutil.rmtree(bengi_dst)
    shutil.copytree(bengi_src, bengi_dst)
    print(f'BENGI: {len(os.listdir(bengi_dst))} files')

# Copy genomic_data config
gdata_src = os.path.join(data_root, 'genomic_data')
gdata_dst = os.path.join(DATA_DIR, 'genomic_data')
if os.path.exists(gdata_src):
    if os.path.exists(gdata_dst):
        shutil.rmtree(gdata_dst)
    shutil.copytree(gdata_src, gdata_dst)
    print(f'genomic_data: copied')

!ls -la {DATA_DIR}/BENGI/

In [ ]:
# Unzip processed .pt files into the correct location
os.chdir(os.path.expanduser('~'))
PROC_DIR = os.path.join(DATA_DIR, 'genomic_data', 'processed')
os.makedirs(PROC_DIR, exist_ok=True)

!unzip -qo processed.zip -d processed_tmp

# Find all .pt files and copy them
pt_files = glob.glob('processed_tmp/**/*.pt', recursive=True)
for f in pt_files:
    shutil.copy2(f, PROC_DIR)
shutil.rmtree('processed_tmp', ignore_errors=True)

print(f'Processed .pt files: {len(os.listdir(PROC_DIR))}')
!ls {PROC_DIR} | head -10

In [ ]:
# Update the JSON config to point to the local processed/ path
import json

json_path = os.path.join(DATA_DIR, 'genomic_data', 'processed',
                         'CTCF_DNase_6histone_local.500.json')
# If JSON is inside genomic_data/ (not processed/), find it
if not os.path.exists(json_path):
    json_candidates = glob.glob(os.path.join(DATA_DIR, 'genomic_data', '**', '*.json'), recursive=True)
    print(f'JSON candidates found: {[os.path.basename(c) for c in json_candidates]}')
    # Must pick the CTCF_DNase file, NOT config_local.json
    for c in json_candidates:
        basename = os.path.basename(c)
        if 'CTCF' in basename and 'local' in basename:
            json_path = c
            break
    else:
        # Fallback: any CTCF file
        for c in json_candidates:
            if 'CTCF' in os.path.basename(c):
                json_path = c
                break
    if not os.path.exists(json_path):
        print('WARNING: No CTCF_DNase JSON config found!')

print(f'Using config: {json_path}')

# Update _location to absolute path
with open(json_path, 'r') as f:
    feat_cfg = json.load(f)

feat_cfg['_location'] = PROC_DIR

updated_json = os.path.join(DATA_DIR, 'genomic_data', 'feats_config.json')
with open(updated_json, 'w') as f:
    json.dump(feat_cfg, f, indent=2)

print(f'Updated config saved to: {updated_json}')
print(f'Cell lines: {[k for k in feat_cfg if k != "_location"]}')

In [ ]:
# Clean up zip files to save disk space
os.chdir(os.path.expanduser('~'))
for zf in ['POCD-KansformerEPI.zip', 'processed.zip', 'kansformer.zip']:
    if os.path.exists(zf):
        os.remove(zf)
        print(f'Removed {zf}')
shutil.rmtree('kansformer_tmp', ignore_errors=True)
print('Cleanup done.')

## Step 2 — Install Dependencies

In [ ]:
os.chdir(PROJECT)
!pip install -q -r requirements.txt

## Step 3 — (Optional) Download hg19 Reference Genome

The DNA sequence branch uses POCD-ND encoding of enhancer/promoter sequences.
Without a reference genome, the model trains with **epigenetic features only**
(dummy N-sequences). To enable the full dual-branch architecture, download hg19:

⏱️ This takes ~5 minutes (~3 GB download + indexing)

In [ ]:
# OPTIONAL — Skip this cell if you want epi-only mode
REF_DIR = os.path.join(DATA_DIR, 'reference')
os.makedirs(REF_DIR, exist_ok=True)

HG19_URL = 'https://hgdownload.soe.ucsc.edu/goldenPath/hg19/bigZips/hg19.fa.gz'
HG19_FA = os.path.join(REF_DIR, 'hg19.fa')

if not os.path.exists(HG19_FA):
    print('Downloading hg19 reference genome...')
    !wget -q --show-progress -O {REF_DIR}/hg19.fa.gz {HG19_URL}
    print('Decompressing...')
    !gunzip {REF_DIR}/hg19.fa.gz
    print('Indexing with samtools...')
    !pip install -q pysam
    !samtools faidx {HG19_FA} || python -c "import pysam; pysam.FastaFile('{HG19_FA}')"
    print(f'Done. Reference at: {HG19_FA}')
else:
    print(f'hg19 already exists at {HG19_FA}')

## Step 4 — Update Config for This Machine

In [ ]:
import yaml

os.chdir(PROJECT)
with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Update paths for Vertex AI
config['paths']['bengi_dir'] = os.path.join(DATA_DIR, 'BENGI')
config['paths']['feats_config'] = updated_json  # from Step 1

# Set reference genome path (empty string = epi-only mode)
if os.path.exists(os.path.join(DATA_DIR, 'reference', 'hg19.fa')):
    config['paths']['ref_genome'] = os.path.join(DATA_DIR, 'reference', 'hg19.fa')
    print('✅ Reference genome enabled — full dual-branch mode')
else:
    config['paths']['ref_genome'] = ''
    print('ℹ️ No reference genome — using epi-only mode (sequence branch gets N-sequences)')

# Training settings for L4 GPU
config['training']['epochs'] = 50
config['training']['lr'] = 0.0001
config['training']['patience'] = 10
config['training']['num_workers'] = 4
config['data']['batch_size'] = 32

with open('configs/config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print('\nConfig updated:')
print(yaml.dump(config, default_flow_style=False))

## Step 5 — Verify Data Pipeline

In [ ]:
os.chdir(PROJECT)
import torch
import glob

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Check data files
bengi_files = sorted(glob.glob(os.path.join(config['paths']['bengi_dir'], '*.tsv*')))
pt_files = glob.glob(os.path.join(PROC_DIR, '*.pt'))
print(f'\nBENGI files: {len(bengi_files)}')
for f in bengi_files:
    print(f'  {os.path.basename(f)}')
print(f'\nProcessed .pt files: {len(pt_files)}')

In [ ]:
# Test-load one BENGI file + .pt features
from src.epi_data_pipeline import EPIGenomicDataset
from src.encoding import POCD_ND_Encoder
from src.dataset import EPIDataset
import numpy as np

# Use just one BENGI file for quick test
test_bengi = bengi_files[0]
print(f'Testing with: {os.path.basename(test_bengi)}')

test_ds = EPIGenomicDataset(
    bengi_paths=[test_bengi],
    feats_config_path=config['paths']['feats_config'],
    feats_order=config['data']['feats_order'],
    seq_len=config['data']['seq_len_bp'],
    bin_size=config['data']['bin_size'],
    enhancer_window=config['data']['enhancer_window'],
    promoter_window=config['data']['promoter_window'],
    ref_genome_path=config['paths'].get('ref_genome', ''),
)

# Quick sample check
sample = test_ds[0]
print(f'\nSample 0:')
print(f'  epi shape: {sample["epi"].shape}')
print(f'  enhancer_seq: {sample["enhancer_seq"][:50]}...')
print(f'  promoter_seq: {sample["promoter_seq"][:50]}...')
print(f'  label: {sample["label"].item()}')
print(f'  dist: {sample["dist"].item():.4f}')

# Test POCD-ND encoding
encoder = POCD_ND_Encoder(k=config['data']['kmer_size'])
dummy_seqs = ["ACGT" * (config['data']['sequence_length'] // 4) for _ in range(50)]
encoder.fit(dummy_seqs, dummy_seqs, config['data']['sequence_length'])

epi_dataset = EPIDataset(config, encoder, source_dataset=test_ds)
item = epi_dataset[0]
print(f'\nEncoded sample:')
print(f'  seq: {item["seq"].shape}')   # (64, seq_len)
print(f'  epi: {item["epi"].shape}')   # (n_feats, n_bins)
print(f'  label: {item["label"]}')     
print(f'\n✅ Pipeline works!')

## Step 6 — Train the Model

You can either run `train.py` directly or use the training loop below for
interactive monitoring with progress bars.

In [ ]:
# Option A: Run train.py directly (simpler)
# !python train.py

In [ ]:
# Option B: Interactive training with progress bars
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import yaml, os, glob, pickle
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm.notebook import tqdm

from src.epi_data_pipeline import EPIGenomicDataset
from src.dataset import EPIDataset
from src.model import Kansformer
from src.encoding import POCD_ND_Encoder
from src.visualize import plot_history

os.chdir(PROJECT)
with open('configs/config.yaml') as f:
    config = yaml.safe_load(f)
os.makedirs(config['paths']['save_dir'], exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Encoder
encoder = POCD_ND_Encoder(k=config['data']['kmer_size'])
np.random.seed(42)
seq_len = config['data']['sequence_length']
dummy = ["ACGT" * (seq_len // 4) for _ in range(50)]
encoder.fit(dummy, dummy, seq_len)
with open(f"{config['paths']['save_dir']}/encoder.pkl", 'wb') as f:
    pickle.dump(encoder, f)
print('Encoder ready.')

In [ ]:
# Load ALL data
bengi_files = sorted(
    glob.glob(os.path.join(config['paths']['bengi_dir'], '*.tsv.gz')) +
    glob.glob(os.path.join(config['paths']['bengi_dir'], '*.tsv'))
)
print(f'Loading {len(bengi_files)} BENGI files...')

genomic_ds = EPIGenomicDataset(
    bengi_paths=bengi_files,
    feats_config_path=config['paths']['feats_config'],
    feats_order=config['data']['feats_order'],
    seq_len=config['data']['seq_len_bp'],
    bin_size=config['data']['bin_size'],
    enhancer_window=config['data']['enhancer_window'],
    promoter_window=config['data']['promoter_window'],
    ref_genome_path=config['paths'].get('ref_genome', '') or None,
)

dataset = EPIDataset(config, encoder, source_dataset=genomic_ds)

# Split by chromosome
valid_chroms = config['training'].get('valid_chroms', ['chr11', 'chr17'])
chroms = genomic_ds.get_chrom_groups()
train_idx = [i for i, c in enumerate(chroms) if c not in valid_chroms]
val_idx = [i for i, c in enumerate(chroms) if c in valid_chroms]

train_set = Subset(dataset, train_idx)
val_set = Subset(dataset, val_idx)
print(f'\nTrain: {len(train_set)}, Val: {len(val_set)} (held-out: {valid_chroms})')

train_loader = DataLoader(train_set, batch_size=config['data']['batch_size'],
                          shuffle=True, num_workers=config['training'].get('num_workers', 4))
val_loader = DataLoader(val_set, batch_size=config['data']['batch_size'],
                        num_workers=config['training'].get('num_workers', 4))

In [ ]:
# Model
model = Kansformer(config).to(device)
optimizer = optim.Adam(model.parameters(), lr=config['training']['lr'])
crit_cls = nn.BCELoss()
crit_reg = nn.MSELoss()

total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {total_p:,} total params, {train_p:,} trainable')

In [ ]:
# Training loop with tqdm
train_hist, val_hist = [], []
best_val_loss = float('inf')
patience_counter = 0
EPOCHS = config['training']['epochs']
PATIENCE = config['training'].get('patience', 10)
GRAD_CLIP = config['training'].get('grad_clip', 1.0)
LAMBDA_DIST = config['training']['lambda_dist']

for epoch in range(EPOCHS):
    # ---- Train ----
    model.train()
    epoch_loss = 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]', leave=False)
    for batch in pbar:
        seq = batch['seq'].float().to(device)
        epi = batch['epi'].float().to(device)
        lbl = batch['label'].float().to(device)
        dst = batch['dist'].float().to(device)
        
        optimizer.zero_grad()
        p_cls, p_reg = model(seq, epi)
        loss = crit_cls(p_cls, lbl) + LAMBDA_DIST * crit_reg(p_reg, dst)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        epoch_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    # ---- Validate ----
    model.eval()
    val_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]', leave=False):
            seq = batch['seq'].float().to(device)
            epi = batch['epi'].float().to(device)
            lbl = batch['label'].float().to(device)
            dst = batch['dist'].float().to(device)
            p_cls, p_reg = model(seq, epi)
            val_loss += (crit_cls(p_cls, lbl) + LAMBDA_DIST * crit_reg(p_reg, dst)).item()
            all_preds.extend(p_cls.cpu().numpy().flatten())
            all_labels.extend(lbl.cpu().numpy().flatten())

    avg_train = epoch_loss / max(len(train_loader), 1)
    avg_val = val_loss / max(len(val_loader), 1)
    train_hist.append(avg_train)
    val_hist.append(avg_val)

    preds_np = np.array(all_preds)
    labels_np = np.array(all_labels)
    acc = accuracy_score(labels_np, (preds_np > 0.5).astype(int))
    try:
        auc = roc_auc_score(labels_np, preds_np)
    except ValueError:
        auc = 0.0

    marker = ''
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), f"{config['paths']['save_dir']}/model_best.pth")
        patience_counter = 0
        marker = ' ⭐ best'
    else:
        patience_counter += 1

    print(f'Epoch {epoch+1}/{EPOCHS} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | '
          f'Acc: {acc:.4f} | AUC: {auc:.4f}{marker}')

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1}')
        break

torch.save(model.state_dict(), f"{config['paths']['save_dir']}/model_final.pth")
print('\nTraining complete!')

## Step 7 — Visualise Training

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(train_hist, label='Train Loss')
ax.plot(val_hist, label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('POCD-KansformerEPI Training')
ax.legend()
plt.tight_layout()
plt.savefig(f"{config['paths']['save_dir']}/loss.png", dpi=150)
plt.show()

## Step 8 — Evaluate & Interpret

In [ ]:
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support,
    confusion_matrix, roc_curve, precision_recall_curve, auc
)

# Load best model
best_model = Kansformer(config).to(device)
best_model.load_state_dict(torch.load(
    f"{config['paths']['save_dir']}/model_best.pth",
    map_location=device, weights_only=True
))
best_model.eval()

# Full validation
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(val_loader, desc='Evaluating'):
        seq = batch['seq'].float().to(device)
        epi = batch['epi'].float().to(device)
        lbl = batch['label'].float().to(device)
        p_cls, _ = best_model(seq, epi)
        all_preds.extend(p_cls.cpu().numpy().flatten())
        all_labels.extend(lbl.cpu().numpy().flatten())

preds = np.array(all_preds)
labels = np.array(all_labels)
pred_binary = (preds > 0.5).astype(int)

acc = accuracy_score(labels, pred_binary)
auroc = roc_auc_score(labels, preds)
prec, rec, f1, _ = precision_recall_fscore_support(labels, pred_binary, average='binary')
pr_precision, pr_recall, _ = precision_recall_curve(labels, preds)
aupr = auc(pr_recall, pr_precision)

print(f'Accuracy:  {acc:.4f}')
print(f'AUROC:     {auroc:.4f}')
print(f'AUPR:      {aupr:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print(f'F1:        {f1:.4f}')
print(f'\nConfusion Matrix:')
print(confusion_matrix(labels, pred_binary))

In [ ]:
# ROC and PR curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

fpr, tpr, _ = roc_curve(labels, preds)
ax1.plot(fpr, tpr, lw=2, label=f'AUROC = {auroc:.3f}')
ax1.plot([0, 1], [0, 1], 'k--', lw=1)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve')
ax1.legend()

ax2.plot(pr_recall, pr_precision, lw=2, label=f'AUPR = {aupr:.3f}')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve')
ax2.legend()

plt.tight_layout()
plt.savefig(f"{config['paths']['save_dir']}/eval_curves.png", dpi=150)
plt.show()

## Step 9 — CAM Interpretation (Single Sample)

In [ ]:
from src.interpretation import Interpreter
from src.visualize import plot_cam

# Pick a positive sample from validation set
pos_indices = [i for i, l in enumerate(labels) if l == 1]
if pos_indices:
    sample_idx = val_idx[pos_indices[0]]
    sample = dataset[sample_idx]
    
    interp = Interpreter(best_model.cpu())
    cam_scores = interp.get_cam(sample['seq'], sample['epi'])
    
    # The sequence is synthetic Ns if no ref genome — show CAM regardless
    plot_cam('N' * len(cam_scores), cam_scores,
             f"{config['paths']['save_dir']}/cam_positive_sample.png")
    print('CAM saved to checkpoints/cam_positive_sample.png')
else:
    print('No positive samples in validation set')

## Step 10 — Download Results

Download the `checkpoints/` folder via JupyterLab File Browser:
- Right-click on `checkpoints/` → Download

Key files:
- `model_best.pth` — best model weights
- `model_final.pth` — final model weights
- `encoder.pkl` — fitted POCD-ND encoder
- `loss.png` — training curves
- `eval_curves.png` — ROC/PR curves

In [ ]:
# List all checkpoints
!ls -lh {config['paths']['save_dir']}/